# Knowledge Sources in CrewAI [Step 2 -- Grounding agents in documents]

> **MLCourse - Agentic AI - CrewAI Advanced Agents**

An LLM without context hallucinates. Knowledge Sources give CrewAI agents
access to external documents -- text files, PDFs, or custom data -- so they
reason over real information instead of inventing it. This notebook covers
how to attach, configure, and query knowledge in CrewAI.

## What you will learn

- `TextKnowledgeSource`: load plain-text files as retrieval context
- `PDFKnowledgeSource`: load PDF documents (when available)
- `crewai.Knowledge`: aggregate multiple sources into one retrievable bundle
- Knowledge at the Agent level vs the Crew level: scoping and precedence
- How knowledge feeds into agent reasoning during task execution

In [ ]:
# === SETUP CELL ===
import os
from pathlib import Path

from dotenv import load_dotenv

# Walk up from cwd until we reach the track root folder "03_agentic_ai".
# This lets the notebook run from any subfolder while finding the shared .env.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Guard Jupyter-only magic so this file stays valid as plain Python too.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1. Create sample knowledge files

Before we can load knowledge, we need files on disk. Below we create two
sample files: a plain-text company FAQ and a technical reference. In
production you would point these at your real documentation, wikis, or data.

In [ ]:
# Define paths for our sample knowledge files.
KNOWLEDGE_DIR = TRACK / "04_crewai" / "data"
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)

# Sample 1: Company FAQ in plain text.
faq_path = KNOWLEDGE_DIR / "company_faq.txt"
faq_path.write_text(
    "Frequently Asked Questions - TechCorp\n"
    "\n"
    "Q: What is TechCorp's main product?\n"
    "A: TechCorp builds AI-powered analytics dashboards for mid-size enterprises.\n"
    "\n"
    "Q: How many employees does TechCorp have?\n"
    "A: TechCorp has approximately 450 employees across three offices.\n"
    "\n"
    "Q: Where are TechCorp's offices located?\n"
    "A: San Francisco (headquarters), London, and Singapore.\n"
    "\n"
    "Q: What programming languages does TechCorp use?\n"
    "A: Python, TypeScript, and Go are the primary languages in the tech stack.\n"
    "\n"
    "Q: Does TechCorp offer remote work?\n"
    "A: Yes, all engineering roles support full remote work within US time zones.\n",
    encoding="utf-8",
)
print(f"Created: {faq_path} ({faq_path.stat().st_size} bytes)")

# Sample 2: Technical reference.
ref_path = KNOWLEDGE_DIR / "tech_reference.txt"
ref_path.write_text(
    "TechCorp Analytics Platform - Technical Reference\n"
    "\n"
    "Architecture: The platform uses a microservices architecture with FastAPI "
    "backends, PostgreSQL for persistence, and Redis for caching.\n"
    "\n"
    "Authentication: JWT tokens with RS256 signing. Tokens expire after 24 hours. "
    "Refresh tokens are valid for 30 days.\n"
    "\n"
    "Rate Limits: Free tier allows 100 requests per minute. Pro tier allows "
    "1000 requests per minute. Enterprise tier has no rate limits.\n"
    "\n"
    "Data Retention: Free tier retains data for 30 days. Pro tier retains data "
    "for 1 year. Enterprise tier retains data indefinitely.\n"
    "\n"
    "SDK: The Python SDK is installed via 'pip install techcorp-sdk'. "
    "The latest stable version is 2.4.1.\n",
    encoding="utf-8",
)
print(f"Created: {ref_path} ({ref_path.stat().st_size} bytes)")

## 2. `TextKnowledgeSource` -- loading plain text files

`TextKnowledgeSource` is the most straightforward knowledge source. It reads
a `.txt` file, chunks it into retrieval-sized segments, and makes those
chunks available to agents during reasoning.

**How chunking works:**
- Text is split into chunks of ~1000 characters by default
- Overlap between chunks preserves context at boundaries
- Each chunk becomes a retrieval unit the agent can surface

**Parameters:**
- `path` (str or Path): path to the text file
- `chunk_size` (int): max characters per chunk (default varies by version)
- `chunk_overlap` (int): character overlap between consecutive chunks

In [ ]:
from crewai import Knowledge, TextKnowledgeSource

# Create a TextKnowledgeSource from our FAQ file.
faq_source = TextKnowledgeSource(path=str(faq_path))

# Wrap it in a Knowledge object -- this is the container CrewAI uses.
faq_knowledge = Knowledge(sources=[faq_source])

print(f"Source type: {type(faq_source).__name__}")
print(f"Knowledge object created with {len(faq_knowledge.sources)} source(s)")
print(f"Knowledge content preview: {str(faq_knowledge)[:120]}...")

## 3. `PDFKnowledgeSource` -- loading PDF documents

`PDFKnowledgeSource` works identically to `TextKnowledgeSource` but reads
PDF files. It requires a PDF parsing library (like PyPDF2) which CrewAI
handles as an optional dependency.

**Important:** This cell is guarded because PDF dependencies may not be
installed in every environment. The pattern is the same regardless of
whether the source is text or PDF -- the `Knowledge` object abstracts it.

In [ ]:
try:
    from crewai import PDFKnowledgeSource

    # Create a PDF source -- point this at any PDF file.
    # For demonstration we create a minimal PDF using reportlab if available.
    pdf_path = KNOWLEDGE_DIR / "sample_doc.pdf"
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.pdfgen import canvas

        c = canvas.Canvas(str(pdf_path), pagesize=letter)
        c.drawString(72, 720, "TechCorp API Documentation")
        c.drawString(72, 700, "Version 2.4.1 - Released 2026-01-15")
        c.drawString(72, 680, "Base URL: https://api.techcorp.com/v2")
        c.drawString(72, 660, "Authentication: Bearer token via Authorization header")
        c.save()
        print(f"Created sample PDF: {pdf_path}")

        pdf_source = PDFKnowledgeSource(path=str(pdf_path))
        pdf_knowledge = Knowledge(sources=[pdf_source])
        print(f"PDF Knowledge created with {len(pdf_knowledge.sources)} source(s)")
    except ImportError:
        print("[skip] reportlab not installed -- cannot create sample PDF")
        print("To enable: pip install reportlab")
        pdf_source = None

except ImportError:
    print("[skip] PDFKnowledgeSource not available in this CrewAI version")
    print("The pattern is identical to TextKnowledgeSource -- swap the class name")
    pdf_source = None

## 4. Multiple sources in one `Knowledge` bundle

Real projects combine multiple documents. `Knowledge` accepts a list of
sources and merges them into a single retrieval context. Agents query the
combined knowledge and get relevant chunks from whichever source matches.

This is how you give an agent access to an entire documentation library:
one `Knowledge` object with many `TextKnowledgeSource` entries.

In [ ]:
# Combine FAQ and technical reference into one Knowledge bundle.
combined_knowledge = Knowledge(
    sources=[
        TextKnowledgeSource(path=str(faq_path)),
        TextKnowledgeSource(path=str(ref_path)),
    ]
)

print(f"Combined knowledge has {len(combined_knowledge.sources)} source(s)")
for i, src in enumerate(combined_knowledge.sources):
    print(f"  Source {i}: {type(src).__name__} -> {getattr(src, 'path', 'N/A')}")

## 5. Knowledge on Agent vs Crew -- scoping matters

CrewAI supports attaching knowledge at two levels:

**Agent-level knowledge:** Each agent carries its own `knowledge` parameter.
Only that agent sees the documents. Use this when different agents need
different information (e.g., a sales agent reads sales docs, an engineering
agent reads technical docs).

**Crew-level knowledge:** Attached to the `Crew` constructor. ALL agents
in the crew share access. Use this for common reference material that
every agent needs (e.g., company policies, project requirements).

**Precedence:** If both are specified, agent-level sources are used alongside
crew-level sources -- they do not override, they merge.

In [ ]:
from crewai import Agent, Task, Crew, Process

# Agent with its own knowledge -- only this agent sees the FAQ.
faq_agent = Agent(
    role="Customer Support Specialist",
    goal="Answer customer questions accurately using the company FAQ.",
    backstory=(
        "You are a helpful support agent who always references the official "
        "FAQ document when answering customer questions. Never guess."
    ),
    knowledge=[faq_source],  # Agent-level: this agent only.
    llm="ollama/llama3.1:8b",
    verbose=False,
    allow_delegation=False,
)

# Agent with NO agent-level knowledge -- relies on crew-level knowledge only.
general_agent = Agent(
    role="Technical Writer",
    goal="Write documentation that is accurate and referenced.",
    backstory=(
        "You are a technical writer who uses the company knowledge base "
        "to ensure all documentation is accurate and up to date."
    ),
    # No knowledge= here -- will use crew-level knowledge instead.
    llm="ollama/llama3.1:8b",
    verbose=False,
    allow_delegation=False,
)

print("Agent-level knowledge: faq_agent has 1 source attached")
print("Crew-level knowledge: will be set on the Crew below")

## 6. Querying knowledge in a crew execution

When a crew runs, agents with knowledge access can retrieve relevant chunks
from their sources. The retrieval is automatic -- you do not write explicit
search queries. CrewAI feeds relevant document chunks into the agent's
context window before the LLM generates its response.

**How it works under the hood:**
1. Agent receives a task description
2. CrewAI embeds the task text and searches knowledge sources for relevant chunks
3. Top-k matching chunks are injected into the agent's system prompt
4. The LLM responds with the retrieved context available

In [ ]:
# Build a crew with crew-level knowledge (shared by all agents).
crew = Crew(
    agents=[faq_agent, general_agent],
    tasks=[
        Task(
            description=(
                "Answer this customer question: 'How many employees does "
                "TechCorp have and where are the offices located?'"
            ),
            expected_output="A direct answer to the customer question with specific details.",
            agent=faq_agent,
        ),
        Task(
            description=(
                "What is the rate limit for the Pro tier of TechCorp's API, "
                "and what is the data retention policy?"
            ),
            expected_output="Specific rate limit and retention details from the knowledge base.",
            agent=general_agent,
        ),
    ],
    process=Process.sequential,
    knowledge=[combined_knowledge],  # Crew-level: shared by all agents.
    verbose=False,
)

print("Crew assembled with 2 agents and crew-level knowledge from 2 sources")
print("Note: general_agent relies entirely on crew-level knowledge")

## 7. Running the crew -- knowledge-grounded answers

Execute the crew and observe that agents produce answers grounded in the
actual document content. Without knowledge, the LLM would either hallucinate
or refuse to answer. With knowledge, it retrieves the relevant chunks and
incorporates them into its response.

In [ ]:
try:
    result = crew.kickoff()
    print("=== Crew Execution Result ===")
    print(result)
except Exception as e:
    print(f"[demo skipped] Crew execution failed: {e}")
    print("Ensure Ollama is running: ollama serve")

## 8. Inspecting what the agent actually sees

CrewAI's knowledge retrieval can be inspected by examining the agent's
context after execution. The `agent.messages` or crew output metadata
reveals which chunks were retrieved. This is critical for debugging:
if the agent hallucinates, check whether the right chunks were retrieved.

In [ ]:
# After crew execution, inspect the agent's internal messages.
# The messages list shows the full conversation including retrieved context.
try:
    if hasattr(faq_agent, 'messages') and faq_agent.messages:
        print("=== Agent Message History ===")
        for i, msg in enumerate(faq_agent.messages):
            role = getattr(msg, 'role', 'unknown')
            content = str(getattr(msg, 'content', msg))[:200]
            print(f"  [{i}] {role}: {content}...")
    else:
        print("[info] Agent message history not available in this CrewAI version")
        print("After execution, check agent.messages or crew output for retrieved chunks")
except Exception as e:
    print(f"[info] Could not inspect messages: {e}")

## 9. Knowledge without a crew -- direct retrieval

You can also use `Knowledge` objects for direct retrieval without running a
full crew. This is useful for building custom RAG pipelines where you want
CrewAI's chunking but not its agent orchestration.

In [ ]:
# Direct knowledge query -- no agent or crew needed.
try:
    # Knowledge objects support direct search/retrieve methods.
    query = "What is the authentication method?"
    print(f"Query: '{query}'")
    print("-" * 50)

    # Try to retrieve directly from the knowledge object.
    if hasattr(combined_knowledge, 'search'):
        results = combined_knowledge.search(query)
        print(f"Direct search returned {len(results)} result(s)")
        for r in results[:3]:
            print(f"  - {str(r)[:120]}...")
    elif hasattr(combined_knowledge, 'retrieve'):
        results = combined_knowledge.retrieve(query)
        print(f"Direct retrieve returned {len(results)} result(s)")
        for r in results[:3]:
            print(f"  - {str(r)[:120]}...")
    else:
        print("[info] Direct search method not available in this version")
        print("Knowledge retrieval happens automatically during crew execution")

except Exception as e:
    print(f"[info] Direct retrieval not available: {e}")
    print("Knowledge is primarily consumed by agents during crew execution")

## Summary and key takeaways

- `TextKnowledgeSource(path=...)` loads plain text files as retrieval context.
- `PDFKnowledgeSource(path=...)` does the same for PDFs (requires PDF parser).
- `Knowledge(sources=[...])` bundles multiple sources into one retrievable unit.
- **Agent-level knowledge**: scoped to one agent -- different agents, different docs.
- **Crew-level knowledge**: shared across all agents in the crew.
- Knowledge retrieval is automatic during crew execution: relevant chunks are
  injected into the agent's context before the LLM responds.
- Without knowledge, agents hallucinate or refuse. With knowledge, they ground
  their answers in actual document content.

**Next up:** notebook 03 covers Memory Systems -- how agents remember across
executions and track entities.